In [7]:
#комплектовщики
import pandas as pd
import os
import glob
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Исправленный путь
folder_path = r'\\vra.local\Root\Public\ОИС\Системы отчетности и анализа данных\Производительность'

# Целевые колонки для анализа
TARGET_COLUMNS = [
    'Отобрано паллетов',
    'Расчёт отобрано упаковок', 
    'Отобрано ШТ'
]

def analyze_specific_columns():
    """Анализ только для трёх целевых колонок"""
    
    print(f"\n{'='*120}")
    print(f"АНАЛИЗ ЦЕЛЕВЫХ КОЛОНОК: {', '.join(TARGET_COLUMNS)}")
    print(f"Папка: {folder_path}")
    print(f"{'='*120}")
    
    results_by_column = {col: [] for col in TARGET_COLUMNS}
    all_files_processed = 0
    files_with_target_columns = 0
    
    # Получаем список всех xls файлов в папке
    xls_files = glob.glob(os.path.join(folder_path, '*.xls'))
    xlsx_files = glob.glob(os.path.join(folder_path, '*.xlsx'))
    all_excel_files = xls_files + xlsx_files
    
    print(f"Найдено файлов: {len(all_excel_files)}")
    
    for file_idx, file_path in enumerate(all_excel_files, 1):
        file_name = os.path.basename(file_path)
        
        # Прогресс
        if file_idx % 10 == 0:
            print(f"Обработано {file_idx}/{len(all_excel_files)} файлов...")
        
        try:
            # Пробуем прочитать файл
            excel_file = pd.ExcelFile(file_path)
            all_files_processed += 1
            
            for sheet_idx, sheet_name in enumerate(excel_file.sheet_names):
                try:
                    # Читаем лист
                    df = pd.read_excel(file_path, sheet_name=sheet_name)
                    
                    # Проверяем наличие целевых колонок
                    found_columns = [col for col in TARGET_COLUMNS if col in df.columns]
                    
                    if found_columns:
                        files_with_target_columns += 1
                        
                        for target_col in found_columns:
                            # Анализируем конкретную колонку
                            col_analysis = analyze_single_column(
                                df[target_col], 
                                file_name, 
                                sheet_name, 
                                target_col
                            )
                            results_by_column[target_col].append(col_analysis)
                        
                except Exception as e:
                    # Пропускаем лист с ошибкой
                    continue
                    
        except Exception as e:
            # Пропускаем файл с ошибкой
            continue
    
    print(f"\n{'='*120}")
    print(f"ОБЩАЯ СТАТИСТИКА:")
    print(f"Всего файлов в папке: {len(all_excel_files)}")
    print(f"Успешно обработано файлов: {all_files_processed}")
    print(f"Файлов с целевыми колонками: {files_with_target_columns}")
    print(f"{'='*120}")
    
    # Выводим результаты для каждой колонки
    for target_col in TARGET_COLUMNS:
        if results_by_column[target_col]:
            print_column_report(results_by_column[target_col], target_col)
        else:
            print(f"\n❌ Колонка '{target_col}' не найдена ни в одном файле.")


In [8]:
def analyze_single_column(column_data, file_name, sheet_name, column_name):
    """Детальный анализ одной колонки"""
    
    # Основная статистика
    total_values = len(column_data)
    non_null = column_data.count()
    null_count = column_data.isnull().sum()
    
    # Анализ типов данных
    type_distribution = {}
    sample_values = []
    problematic_cases = []
    
    for val in column_data.dropna().head(500):  # Анализируем до 500 значений
        if pd.isna(val):
            continue
            
        # Определяем тип
        if isinstance(val, (int, np.integer)):
            val_type = 'int'
        elif isinstance(val, (float, np.floating)):
            val_type = 'float'
        elif isinstance(val, str):
            val_str = str(val).strip()
            
            # Детальная классификация строк
            if val_str == '':
                val_type = 'empty_string'
            else:
                # Пробуем определить числовую строку
                try:
                    # Убираем пробелы и заменяем запятые на точки
                    clean_val = val_str.replace(' ', '').replace(',', '.')
                    
                    # Пробуем как float
                    float_val = float(clean_val)
                    
                    # Проверяем, целое ли это
                    if float_val.is_integer():
                        val_type = 'string_int'
                    else:
                        val_type = 'string_float'
                        
                    # Сохраняем преобразованное значение
                    val = float_val
                        
                except (ValueError, AttributeError):
                    # Проверяем, это ли процент или единица измерения
                    if '%' in val_str:
                        val_type = 'percentage_string'
                    elif any(unit in val_str.lower() for unit in ['шт', 'уп', 'пал', 'кг', 'г']):
                        val_type = 'unit_string'
                    else:
                        val_type = 'text'
        elif isinstance(val, (pd.Timestamp, datetime)):
            val_type = 'datetime'
        elif isinstance(val, bool):
            val_type = 'bool'
        else:
            val_type = 'other'
        
        type_distribution[val_type] = type_distribution.get(val_type, 0) + 1
        
        # Сохраняем примеры проблемных значений
        if val_type.startswith('string_') and 'int' not in val_type and 'float' not in val_type:
            if len(problematic_cases) < 5:
                problematic_cases.append({
                    'original': str(val)[:100],
                    'type': val_type
                })
        
        # Сохраняем несколько примеров
        if len(sample_values) < 5:
            sample_values.append(str(val)[:50])
    
    # Рассчитываем проценты
    type_percentages = {}
    for type_name, count in type_distribution.items():
        percentage = (count / non_null * 100) if non_null > 0 else 0
        type_percentages[type_name] = percentage
    
    return {
        'file': file_name,
        'sheet': sheet_name,
        'column': column_name,
        'total_values': total_values,
        'non_null': non_null,
        'null_count': null_count,
        'null_percentage': (null_count / total_values * 100) if total_values > 0 else 0,
        'type_distribution': type_distribution,
        'type_percentages': type_percentages,
        'sample_values': sample_values[:5],
        'problematic_cases': problematic_cases,
        'pandas_dtype': str(column_data.dtype)
    }


In [12]:
def print_column_report(column_results, column_name):
    """Выводит отчет по одной колонке"""
    
    print(f"\n{'='*120}")
    print(f"ОТЧЕТ ПО КОЛОНКЕ: {column_name}")
    print(f"Найдено в {len(column_results)} файлах/листах")
    print(f"{'='*120}")
    
    # Сводная статистика по типам
    print(f"\n📊 СВОДНАЯ СТАТИСТИКА ПО ТИПАМ ДАННЫХ:")
    
    all_types = {}
    for result in column_results:
        for type_name, count in result['type_distribution'].items():
            all_types[type_name] = all_types.get(type_name, 0) + count
    
    if all_types:
        # Сортируем по количеству
        sorted_types = sorted(all_types.items(), key=lambda x: x[1], reverse=True)
        
        total_records = sum(all_types.values())
        print(f"Всего проанализировано записей: {total_records:,}")
        print(f"Распределение типов:")
        
        for type_name, count in sorted_types:
            percentage = (count / total_records * 100) if total_records > 0 else 0
            print(f"  {type_name:20} - {count:8,} записей ({percentage:6.2f}%)")
    
    # Детальные результаты по файлам
    print(f"\n📋 ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ ПО ФАЙЛАМ:")
    
    for i, result in enumerate(column_results[:20], 1):  # Показываем первые 20
        print(f"\n{i:3}. Файл: {result['file']}")
        print(f"     Лист: {result['sheet']}")
        print(f"     Всего значений: {result['total_values']:,}")
        print(f"     Непустых: {result['non_null']:,} ({result['null_percentage']:.1f}% пустых)")
        print(f"     Pandas dtype: {result['pandas_dtype']}")
        
        if result['type_distribution']:
            print(f"     Типы в этом файле:")
            for type_name, count in sorted(result['type_distribution'].items(), 
                                         key=lambda x: x[1], reverse=True):
                percentage = result['type_percentages'][type_name]
                print(f"       {type_name:20} - {count:6,} ({percentage:5.1f}%)")
        
        if result['sample_values']:
            print(f"     Примеры значений: {result['sample_values']}")
        
        if result['problematic_cases']:
            print(f"     ⚠️  Проблемные значения (строки, которые можно преобразовать в числа):")
            for case in result['problematic_cases']:
                print(f"       '{case['original']}' → тип: {case['type']}")
    
    # Если файлов больше 20, покажем только статистику
    if len(column_results) > 20:
        print(f"\n... и еще {len(column_results) - 20} файлов с этой колонкой")
    
    # Анализ проблемных случаев
    print(f"\n🔍 АНАЛИЗ ПРОБЛЕМНЫХ СЛУЧАЕВ:")
    
    # Собираем информацию о проблемных файлах
    files_with_mixed_types = []
    files_with_string_numbers = []
    
    for result in column_results:
        types = list(result['type_distribution'].keys())
        
        # Проверяем смешанные типы
        if len(types) > 1:
            files_with_mixed_types.append({
                'file': result['file'],
                'sheet': result['sheet'],
                'types': types,
                'distribution': result['type_distribution'],
                'percentages': result['type_percentages']
            })
            
        # Проверяем строковые числа
        string_num_types = [t for t in types if t.startswith('string_')]
        if string_num_types:
            files_with_string_numbers.append({
                'file': result['file'],
                'sheet': result['sheet'],
                'string_types': string_num_types,
                'distribution': result['type_distribution'],
                'percentages': result['type_percentages']
            })
    
    # Выводим информацию о файлах со смешанными типами
    if files_with_mixed_types:
        print(f"Файлов со смешанными типами данных: {len(files_with_mixed_types)}/{len(column_results)}")
        print(f"Файлы со смешанными типами:")
        
        for i, file_info in enumerate(files_with_mixed_types, 1):
            print(f"\n{i}. Файл: {file_info['file']}")
            print(f"   Лист: {file_info['sheet']}")
            print(f"   Найденные типы: {', '.join(file_info['types'])}")
            
            print(f"   Распределение типов:")
            for type_name, count in sorted(file_info['distribution'].items(), key=lambda x: x[1], reverse=True):
                percentage = file_info['percentages'][type_name]
                print(f"     {type_name:20} - {count:6,} записей ({percentage:5.1f}%)")
    else:
        print(f"Файлов со смешанными типами данных: 0/{len(column_results)}")
        print(f"   ✓ Все файлы имеют единообразные типы данных в этой колонке.")
    
    # Выводим информацию о файлах с числами в виде строк
    if files_with_string_numbers:
        print(f"\nФайлов с числами в виде строк: {len(files_with_string_numbers)}/{len(column_results)}")
        print(f"Файлы с числами в виде строк:")
        
        for i, file_info in enumerate(files_with_string_numbers, 1):
            print(f"\n{i}. Файл: {file_info['file']}")
            print(f"   Лист: {file_info['sheet']}")
            print(f"   Строковые числовые типы: {', '.join(file_info['string_types'])}")
            
            # Показываем только строковые числовые типы
            print(f"   Распределение строковых числовых типов:")
            total_string_nums = 0
            for type_name in file_info['string_types']:
                count = file_info['distribution'].get(type_name, 0)
                percentage = file_info['percentages'].get(type_name, 0)
                total_string_nums += count
                print(f"     {type_name:20} - {count:6,} записей ({percentage:5.1f}%)")
            
            # Общая статистика по файлу
            print(f"   Всего записей в файле: {sum(file_info['distribution'].values()):,}")
            print(f"   Из них строковых чисел: {total_string_nums:,} ({total_string_nums/sum(file_info['distribution'].values())*100:.1f}%)")
    else:
        print(f"\nФайлов с числами в виде строк: 0/{len(column_results)}")
        print(f"   ✓ Все числовые данные хранятся в правильном формате.")
    
    # Рекомендации с указанием конкретных файлов
    if files_with_string_numbers:
        print(f"\n💡 РЕКОМЕНДАЦИЯ: {len(files_with_string_numbers)} файлов содержат числа в виде строк.")
        print("   Проблемные файлы:")
        for file_info in files_with_string_numbers:
            print(f"     - {file_info['file']} (лист: {file_info['sheet']})")
        print("   Рекомендуется преобразовать их в числовой формат для корректных вычислений.")
    
    if files_with_mixed_types:
        print(f"\n⚠️  ВНИМАНИЕ: {len(files_with_mixed_types)} файлов содержат смешанные типы данных.")
        print("   Проблемные файлы:")
        for file_info in files_with_mixed_types:
            print(f"     - {file_info['file']} (лист: {file_info['sheet']})")
        print("   Это может привести к ошибкам при обработке данных.")
        
        # Дополнительные рекомендации
        print(f"\n🔧 КАК ИСПРАВИТЬ:")
        print("   1. Откройте проблемные файлы в Excel")
        print("   2. Найдите указанные листы и колонки")
        print("   3. Преобразуйте все значения в один тип (например, все в числа)")
        print("   4. Сохраните файлы")
        print("   5. Повторите анализ для проверки")

In [10]:
def quick_column_summary():
    """Быстрый анализ только целевых колонок"""
    
    print(f"\n{'='*100}")
    print(f"БЫСТРЫЙ АНАЛИЗ КОЛОНОК: {', '.join(TARGET_COLUMNS)}")
    print(f"{'='*100}")
    
    # Получаем список всех xls файлов в папке
    xls_files = glob.glob(os.path.join(folder_path, '*.xls'))
    xlsx_files = glob.glob(os.path.join(folder_path, '*.xlsx'))
    all_excel_files = xls_files + xlsx_files
    
    summary = {col: {'count': 0, 'files': []} for col in TARGET_COLUMNS}
    
    for file_path in all_excel_files[:50]:  # Анализируем первые 50 файлов для скорости
        file_name = os.path.basename(file_path)
        
        try:
            excel_file = pd.ExcelFile(file_path)
            
            for sheet_name in excel_file.sheet_names[:2]:  # Первые 2 листа
                try:
                    # Читаем только заголовки для скорости
                    df = pd.read_excel(file_path, sheet_name=sheet_name, nrows=0)
                    
                    for target_col in TARGET_COLUMNS:
                        if target_col in df.columns:
                            summary[target_col]['count'] += 1
                            summary[target_col]['files'].append({
                                'file': file_name,
                                'sheet': sheet_name
                            })
                            
                except:
                    continue
                    
        except:
            continue
    
    # Выводим быстрый отчет
    for target_col in TARGET_COLUMNS:
        print(f"\n{target_col}:")
        print(f"  Найдено в {summary[target_col]['count']} файлах/листах")
        
        if summary[target_col]['files']:
            print(f"  Примеры файлов:")
            for file_info in summary[target_col]['files'][:5]:
                print(f"    - {file_info['file']} ({file_info['sheet']})")
            
            if len(summary[target_col]['files']) > 5:
                print(f"    ... и еще {len(summary[target_col]['files']) - 5}")


In [14]:
def export_to_excel(results_by_column):
    """Экспорт результатов в Excel"""
    
    if not any(results_by_column.values()):
        print("Нет данных для экспорта.")
        return
    
    with pd.ExcelWriter('target_columns_analysis.xlsx', engine='openpyxl') as writer:
        for col_name, col_results in results_by_column.items():
            if col_results:
                # Создаем DataFrame для экспорта
                export_data = []
                
                for result in col_results:
                    # Формируем строку для экспорта
                    row = {
                        'Файл': result['file'],
                        'Лист': result['sheet'],
                        'Колонка': result['column'],
                        'Всего_значений': result['total_values'],
                        'Непустых': result['non_null'],
                        'Пустых': result['null_count'],
                        'Процент_пустых': result['null_percentage'],
                        'Pandas_тип': result['pandas_dtype']
                    }
                    
                    # Добавляем типы данных
                    for type_name, count in result['type_distribution'].items():
                        row[f'Тип_{type_name}'] = count
                        row[f'Процент_{type_name}'] = result['type_percentages'].get(type_name, 0)
                    
                    export_data.append(row)
                
                df_export = pd.DataFrame(export_data)
                df_export.to_excel(writer, sheet_name=col_name[:31], index=False)
        
        print("Результаты экспортированы в target_columns_analysis.xlsx")

# Главная функция
if __name__ == "__main__":
    # Выберите нужную функцию:
    
    # 1. Полный анализ трех целевых колонок
    analyze_specific_columns()
    
    # 2. Быстрый анализ (только поиск колонок)
    # quick_column_summary()
    
    # 3. Можно также сохранить результаты в Excel
    # Для этого нужно модифицировать analyze_specific_columns() 
    # чтобы она возвращала results_by_column


АНАЛИЗ ЦЕЛЕВЫХ КОЛОНОК: Отобрано паллетов, Расчёт отобрано упаковок, Отобрано ШТ
Папка: \\vra.local\Root\Public\ОИС\Системы отчетности и анализа данных\Производительность
Найдено файлов: 195
Обработано 10/195 файлов...
Обработано 20/195 файлов...
Обработано 30/195 файлов...
Обработано 40/195 файлов...
Обработано 50/195 файлов...
Обработано 60/195 файлов...
Обработано 70/195 файлов...
Обработано 80/195 файлов...
Обработано 90/195 файлов...
Обработано 100/195 файлов...
Обработано 110/195 файлов...
Обработано 120/195 файлов...
Обработано 130/195 файлов...
Обработано 140/195 файлов...
Обработано 150/195 файлов...
Обработано 160/195 файлов...
Обработано 170/195 файлов...
Обработано 180/195 файлов...
Обработано 190/195 файлов...

ОБЩАЯ СТАТИСТИКА:
Всего файлов в папке: 195
Успешно обработано файлов: 195
Файлов с целевыми колонками: 780

ОТЧЕТ ПО КОЛОНКЕ: Отобрано паллетов
Найдено в 780 файлах/листах

📊 СВОДНАЯ СТАТИСТИКА ПО ТИПАМ ДАННЫХ:
Всего проанализировано записей: 21,714
Распределение 

In [13]:
def detailed_problem_analysis():
    """Детальный анализ проблемных файлов"""
    
    print(f"\n{'='*120}")
    print(f"ДЕТАЛЬНЫЙ АНАЛИЗ ПРОБЛЕМНЫХ ФАЙЛОВ")
    print(f"Анализируемые колонки: {', '.join(TARGET_COLUMNS)}")
    print(f"{'='*120}")
    
    # Получаем список всех xls файлов в папке
    xls_files = glob.glob(os.path.join(folder_path, '*.xls'))
    xlsx_files = glob.glob(os.path.join(folder_path, '*.xlsx'))
    all_excel_files = xls_files + xlsx_files
    
    problematic_files_summary = []
    
    for file_path in all_excel_files:
        file_name = os.path.basename(file_path)
        file_problems = []
        
        try:
            excel_file = pd.ExcelFile(file_path)
            
            for sheet_name in excel_file.sheet_names:
                try:
                    df = pd.read_excel(file_path, sheet_name=sheet_name)
                    
                    # Проверяем каждую целевую колонку
                    for target_col in TARGET_COLUMNS:
                        if target_col in df.columns:
                            col_analysis = analyze_single_column(
                                df[target_col], 
                                file_name, 
                                sheet_name, 
                                target_col
                            )
                            
                            types = list(col_analysis['type_distribution'].keys())
                            
                            # Если есть проблемы
                            if len(types) > 1:
                                file_problems.append({
                                    'sheet': sheet_name,
                                    'column': target_col,
                                    'types': types,
                                    'distribution': col_analysis['type_distribution'],
                                    'sample_values': col_analysis['sample_values']
                                })
                            
                except Exception as e:
                    continue
                    
        except Exception as e:
            continue
        
        # Если в файле найдены проблемы
        if file_problems:
            problematic_files_summary.append({
                'file': file_name,
                'problems': file_problems
            })
    
    # Выводим результаты
    if problematic_files_summary:
        print(f"\nНайдено проблемных файлов: {len(problematic_files_summary)}")
        print(f"{'='*120}")
        
        for file_info in problematic_files_summary:
            print(f"\n📁 ФАЙЛ: {file_info['file']}")
            print(f"   Обнаружено проблем: {len(file_info['problems'])}")
            print(f"   {'─'*50}")
            
            for i, problem in enumerate(file_info['problems'], 1):
                print(f"\n   Проблема {i}:")
                print(f"     Лист: {problem['sheet']}")
                print(f"     Колонка: {problem['column']}")
                print(f"     Найденные типы: {', '.join(problem['types'])}")
                
                print(f"     Распределение:")
                total_records = sum(problem['distribution'].values())
                for type_name, count in sorted(problem['distribution'].items(), 
                                              key=lambda x: x[1], reverse=True):
                    percentage = (count / total_records * 100) if total_records > 0 else 0
                    print(f"       {type_name:20} - {count:6,} ({percentage:5.1f}%)")
                
                if problem['sample_values']:
                    print(f"     Примеры значений: {problem['sample_values']}")
            
            print(f"   {'─'*50}")
            print(f"   💡 РЕКОМЕНДАЦИИ ДЛЯ ЭТОГО ФАЙЛА:")
            print(f"      1. Проверьте листы: {', '.join(set(p['sheet'] for p in file_info['problems']))}")
            print(f"      2. Проверьте колонки: {', '.join(set(p['column'] for p in file_info['problems']))}")
            print(f"      3. Убедитесь, что все значения в проблемных колонках одного типа")
    
    else:
        print(f"\n✅ Проблемных файлов не обнаружено!")
        print(f"   Все {len(all_excel_files)} файлов имеют единообразные типы данных в целевых колонках.")
    
    return problematic_files_summary